In [3]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.circuit import QuantumCircuit
from qiskit import transpile

qc = QuantumCircuit(1, 1)
qc.x(0)
qc.measure(0, 0)
    
# Build noise model
noise_model = NoiseModel()
error = depolarizing_error(0.75, 1)
noise_model.add_all_qubit_quantum_error(
    error, ['u1', 'u2', 'u3', 'u', 'p', 'sx', 'x', 'rx', 'ry', 'rz']
)

# Noiseless
sim_ideal = AerSimulator()
result_ideal = sim_ideal.run(transpile(qc, sim_ideal), shots=1024).result()
print("Noiseless     :", result_ideal.get_counts())

# Noisy
sim_noisy = AerSimulator(noise_model=noise_model)
result_noisy = sim_noisy.run(transpile(qc, sim_noisy), shots=1024).result()
print("Noisy p=0.75  :", result_noisy.get_counts())

sim_check = AerSimulator()
qc_t = transpile(qc, sim_check)
print("Basis gates used:", qc_t.count_ops())

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json

from qiskit.circuit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem

print("All imports OK ✅")

p_true = 0.1  ########################################################################################################

class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        q_k = QuantumCircuit(1)
        q_k.ry(2 * k * self._theta_p, 0)
        return q_k

problem = EstimationProblem(
    state_preparation=BernoulliA(p_true),
    grover_operator=BernoulliQ(p_true),
    objective_qubits=[0],
)
print("Problem defined ✅")
    

# ══════════════════════════════════════════════════════════════════════════════
# NOISE MODEL BUILDER
# ══════════════════════════════════════════════════════════════════════════════
# All standard single-qubit basis/native gates Qiskit might emit
SINGLE_QUBIT_GATES = [
    'id', 'x', 'y', 'z', 'h', 's', 'sdg', 't', 'tdg',
    'sx', 'sxdg', 'rx', 'ry', 'rz',
    'u', 'u1', 'u2', 'u3', 'p'
]

# Physical ceiling for 1-qubit depolarizing_error:
#   E(rho) = (1-λ)ρ + (λ/4)(ρ + XρX + YρY + ZρZ)
#   => c_I = 1 - 3λ/4, c_X = c_Y = c_Z = λ/4
#   CP requires all c >= 0  =>  λ ∈ [0, 4/3]
#   λ = 1 is full depolarization (output = I/2); λ ∈ (1, 4/3] is
#   "over-rotation" with no interpretable "more noise" meaning here.
# For a noise-frontier sweep, the meaningful physical range is λ ∈ [0, 1].
MAX_DEPOL_LAMBDA = 1.0          # λ ≤ 1: physically meaningful range

def build_noisy_sampler(lambda_depol, n_shots):
    if lambda_depol > 0.0:
        lam = min(lambda_depol, MAX_DEPOL_LAMBDA)
        error_1q = depolarizing_error(lam, 1)

        noise_model = NoiseModel()
        noise_model.add_all_qubit_quantum_error(error_1q, SINGLE_QUBIT_GATES)

        sampler = AerSampler(
            backend_options={"noise_model": noise_model},
            run_options={"shots": n_shots}
        )
    else:
        sampler = AerSampler(run_options={"shots": n_shots})
    return sampler


Noiseless     : {'1': 1024}
Noisy p=0.75  : {'1': 619, '0': 405}
Basis gates used: OrderedDict({'x': 1, 'measure': 1})
All imports OK ✅
Problem defined ✅


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
from joblib import Parallel, delayed

from qiskit.circuit import QuantumCircuit
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem

print("All imports OK ✅")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
p_true = 0.1
ALPHA = 0.05
R = 100           # Number of Bernoulli trials per point (from image)
N_SHOTS = 1024   # Shot budget (from image)
N_JOBS = -1      # Use all available CPU cores for the grid sweep

# ══════════════════════════════════════════════════════════════════════════════
# PROBLEM DEFINITION
# ══════════════════════════════════════════════════════════════════════════════
class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        q_k = QuantumCircuit(1)
        q_k.ry(2 * k * self._theta_p, 0)
        return q_k

problem = EstimationProblem(
    state_preparation=BernoulliA(p_true),
    grover_operator=BernoulliQ(p_true),
    objective_qubits=[0],
)
print("Problem defined ✅")

# ══════════════════════════════════════════════════════════════════════════════
# NOISE MODEL BUILDER
# ══════════════════════════════════════════════════════════════════════════════
SINGLE_QUBIT_GATES = [
    'id', 'x', 'y', 'z', 'h', 's', 'sdg', 't', 'tdg',
    'sx', 'sxdg', 'rx', 'ry', 'rz',
    'u', 'u1', 'u2', 'u3', 'p'
]
MAX_DEPOL_LAMBDA = 1.0 

def build_noisy_sampler(lambda_depol, n_shots):
    if lambda_depol > 0.0:
        lam = min(lambda_depol, MAX_DEPOL_LAMBDA)
        error_1q = depolarizing_error(lam, 1)
        noise_model = NoiseModel()
        noise_model.add_all_qubit_quantum_error(error_1q, SINGLE_QUBIT_GATES)
        sampler = AerSampler(
            backend_options={"noise_model": noise_model},
            run_options={"shots": n_shots}
        )
    else:
        sampler = AerSampler(run_options={"shots": n_shots})
    return sampler

# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
def single_run(p_noise, eps, shots):
    """One IQAE run. Returns 1 (success) or 0 (failure)."""
    try:
        sampler = build_noisy_sampler(p_noise, shots)
        iae = IterativeAmplitudeEstimation(
            epsilon_target=eps,
            alpha=ALPHA,
            sampler=sampler,
            confint_method="beta" 
        )
        result = iae.estimate(problem)
        return 1 if abs(result.estimation_processed - p_true) <= eps else 0
    except Exception as e:
        return 0

def probe(eps, p):
    """Run R independent IQAE runs sequentially for a specific grid point."""
    k = sum(single_run(p, eps, N_SHOTS) for _ in range(R))
    s_hat = k / R
    return s_hat, N_SHOTS

def evaluate_grid_point(eps, noise):
    """Wrapper for joblib to return coordinates along with the result."""
    s_hat, n_shots = probe(eps, noise)
    return eps, noise, s_hat

# ══════════════════════════════════════════════════════════════════════════════
# 2D GRID SWEEP
# ══════════════════════════════════════════════════════════════════════════════
# Grid resolution settings (Adjust STEPS if you want a denser/faster plot)
EPS_MIN, EPS_MAX, EPS_STEPS = 0.005, 0.1, 50 
NOISE_MIN, NOISE_MAX, NOISE_STEPS = 0.0, 0.45, 50

eps_values = np.linspace(EPS_MIN, EPS_MAX, EPS_STEPS)
noise_values = np.linspace(NOISE_MIN, NOISE_MAX, NOISE_STEPS)
grid_points = list(itertools.product(eps_values, noise_values))

print(f"Starting 2D sweep over {len(grid_points)} grid points...")

# Run the sweep in parallel
grid_results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(evaluate_grid_point)(e, n) for e, n in grid_points
)

print(f"{'═'*60}\n ✅ 2D Grid sweep complete\n{'═'*60}")

# ══════════════════════════════════════════════════════════════════════════════
# PLOTTING
# ══════════════════════════════════════════════════════════════════════════════


import re
import numpy as np

def parse_iqae_ceiling_file(filepath):
    """
    Parse an IQAE ceiling-mode frontier data export file.
    Extracts the first 5 columns: epsilon, p_star, uncertainty, s_hat, n_shots

    Returns:
        epsilon      : np.ndarray
        p_star       : np.ndarray
        uncertainty  : np.ndarray
        s_hat        : np.ndarray
        n_shots      : np.ndarray (int)
    """
    epsilons      = []
    p_stars       = []
    uncertainties = []
    s_hats        = []
    n_shots       = []

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:  # ← fix here
        lines = f.readlines()

    in_data_section = False

    for line in lines:
        line = line.strip()

        # ── Detect start of data section ──────────────────────────────────────
        if line.startswith("epsilon | p_star"):
            in_data_section = True
            continue  # skip the header row

        # ── Skip separator line ───────────────────────────────────────────────
        if in_data_section and set(line) <= set("-"):
            continue

        # ── Parse data rows ───────────────────────────────────────────────────
        if in_data_section and "|" in line:
            parts = [p.strip() for p in line.split("|")]
            epsilons.append(float(parts[0]))
            p_stars.append(float(parts[1]))
            uncertainties.append(float(parts[2]))
            s_hats.append(float(parts[3]))
            n_shots.append(int(parts[4]))

    return (
        np.array(epsilons),
        np.array(p_stars),
        np.array(uncertainties),
        np.array(s_hats),
        np.array(n_shots),
    )


# ── Usage ─────────────────────────────────────────────────────────────────────
ALPHA = 0.05
epsilon, p_star, p_uncertainty, s_hat, n_shots = parse_iqae_ceiling_file(
    "Data/adaptive_searchCP_Nmax, a = 0.1.txt"
) 

plot_eps = [r[0] for r in grid_results]
plot_noise = [r[1] for r in grid_results]
plot_shat = [r[2] for r in grid_results]

plt.figure(figsize=(12, 6))

# Heatmap scatter plot
sc = plt.scatter(plot_eps, plot_noise, c=plot_shat, cmap='RdYlGn', 
                 vmin=0.0, vmax=1.0, s=40, alpha=0.9, marker='o')

sc = plt.scatter(epsilon, p_star, vmin=0.0, vmax=1.0, s=40, alpha=0.9, marker='o')

# Colorbar
cbar = plt.colorbar(sc, label="Success Rate $\hat{s}$")
cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])

# Extract and plot the empirical boundary (where success rate transitions below 0.5)
boundary_eps = []
boundary_noise = []

for e in eps_values:
    e_points = [(n, s) for (eps, n, s) in grid_results if eps == e]
    successful_noises = [n for n, s in e_points if s >= 0.5]
    if successful_noises:
        boundary_eps.append(e)
        boundary_noise.append(max(successful_noises))

# Overlay the empirical boundary markers
plt.plot(boundary_eps, boundary_noise, color='gold', marker='o', 
         linestyle='None', markersize=7, markerfacecolor='none', 
         markeredgewidth=1.5, label='Empirical boundary ($\hat{s} \sim 0.5$)')

# Formatting
plt.title(f"IQAE Noise Threshold Frontier — Shots = {N_SHOTS}\n"
          f"Colour = success rate $\hat{{s}}$ | Each point = {R} Bernoulli trials")
plt.xlabel("$\epsilon$ — Target Precision")
plt.ylabel("Depolarizing Noise Level p")

plt.xlim(EPS_MIN - 0.002, EPS_MAX + 0.002)
plt.ylim(NOISE_MIN - 0.01, NOISE_MAX + 0.01)

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='upper left', framealpha=0.9)
plt.tight_layout()

plt.show()

All imports OK ✅
Problem defined ✅
Starting 2D sweep over 2500 grid points...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  4.7min
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done  88 tasks      | elapsed:  8.1min
[Parallel(n_jobs=-1)]: Done 105 tasks      | elapsed:  9.8min
[Parallel(n_jobs=-1)]: Done 122 tasks      | elapsed: 11.4min
[Parallel(n_jobs=-1)]: Done 141 tasks      | elapsed: 13.0min
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed: 14.5min
[Parallel(n_jobs=-1)]: Done 181 tasks      | elapsed: 15.1min
[Parallel(n_jobs=-1)]: Done 202 tasks      | elapsed: 15.9min


KeyboardInterrupt: 